# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theritik08/Ritik-Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb, os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
con = duckdb.connect()

# List what files/tables exist in the warehouse
files = con.sql("""
    SELECT * FROM glob('hf://datasets/FlyRank/internship-warehouse/**')
""").df()
print(files)

                                                 file
0   hf://datasets/FlyRank/internship-warehouse/.gi...
1   hf://datasets/FlyRank/internship-warehouse/REA...
2   hf://datasets/FlyRank/internship-warehouse/dim...
3   hf://datasets/FlyRank/internship-warehouse/dim...
4   hf://datasets/FlyRank/internship-warehouse/fac...
5   hf://datasets/FlyRank/internship-warehouse/fac...
6   hf://datasets/FlyRank/internship-warehouse/fac...
7   hf://datasets/FlyRank/internship-warehouse/fac...
8   hf://datasets/FlyRank/internship-warehouse/fac...
9   hf://datasets/FlyRank/internship-warehouse/fac...
10  hf://datasets/FlyRank/internship-warehouse/fac...
11  hf://datasets/FlyRank/internship-warehouse/fac...
12  hf://datasets/FlyRank/internship-warehouse/fac...
13  hf://datasets/FlyRank/internship-warehouse/fac...
14  hf://datasets/FlyRank/internship-warehouse/fac...
15  hf://datasets/FlyRank/internship-warehouse/fac...
16  hf://datasets/FlyRank/internship-warehouse/fac...
17  hf://datasets/FlyRank/in

In [2]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
print(files.to_string())

                                                                                                      file
0                                                hf://datasets/FlyRank/internship-warehouse/.gitattributes
1                                                     hf://datasets/FlyRank/internship-warehouse/README.md
2                                           hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
3                                           hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
4   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
5   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
6   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
7   hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
8   hf://datasets/FlyRank/internship-

In [4]:
import duckdb
from google.colab import userdata

# Create a fresh connection
con = duckdb.connect()

# Install and load httpfs extension (required for hf:// paths)
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")

# Register the HF token as a DuckDB secret (proper auth method)
hf_token = userdata.get('HF_TOKEN')
con.sql(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{hf_token}'
    );
""")

# Test the connection
test = con.sql("""
    DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' LIMIT 1
""").df()
print(test)

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [5]:
# Schema for fact_content_daily_performance (the main performance table)
fact_schema = con.sql("""
    DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet' LIMIT 1
""").df()
print("=== fact_content_daily_performance columns ===")
print(fact_schema)

# Schema for dim_clients
clients_schema = con.sql("""
    DESCRIBE SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet' LIMIT 1
""").df()
print("\n=== dim_clients columns ===")
print(clients_schema)

=== fact_content_daily_performance columns ===
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users 

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row =** one content page's search performance for one calendar
day (content_hash_id × report_date grain), from
`fact_content_daily_performance`.

**Time window:** month = '2026-03' (mid-panel month partition —
avoiding the sealed final month 2026-06 and the `_sample` file, which
is exactly June 2026).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label:** CTR, computed as gsc_clicks / gsc_impressions (from
`fact_content_daily_performance`).

**Features:** gsc_avg_position, word_count, char_count, content_type,
main_intent, search_volume, competition (from `dim_content`, joined
on content_hash_id).

**Context (for joins/validation, not fed to model):** client_hash_id,
content_hash_id, report_date, month.

**Excluded (with why):** gsc_clicks alone, ga4_pageviews, ga4_sessions,
sessions_organic — excluded because they are directly derived from or
tightly coupled to the CTR label itself; including them as features
would leak the outcome into the inputs.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# Query 1: Grain check — is it really one row per content_hash_id per day?
grain_check = con.sql("""
    SELECT content_hash_id, report_date, COUNT(*) as n
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()
print(f"Rows violating 1-row-per-content-per-day grain: {len(grain_check)}")

# Query 2: Row count and date span for this slice
counts = con.sql("""
    SELECT COUNT(*) as total_rows,
           MIN(report_date) as min_date,
           MAX(report_date) as max_date
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()
print(counts)

# Query 3: Availability filter — how many rows survive gsc_data_available IS TRUE?
availability = con.sql("""
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as available_rows
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()
print(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating 1-row-per-content-per-day grain: 0
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  available_rows
0     9841378       3611061.0


1. **gsc_avg_position** — available at decision moment because it's
   measured directly from that day's search data, already observed.
2. **word_count** — available because it's static content metadata,
   known before and independent of any given day's performance.
3. **content_type** — same as above, static metadata.
4. **main_intent** — static keyword-level metadata, known ahead of time.
5. **search_volume** — known independently of this content's own
   performance (it's a keyword-level external signal, not derived
   from this page's clicks/impressions).

In [7]:
# Build a 5-feature frame, joining fact + dim_content, filtered to available rows
features_df = con.sql("""
    SELECT
        f.content_hash_id,
        f.report_date,
        f.gsc_avg_position,
        d.word_count,
        d.content_type,
        d.main_intent,
        d.search_volume,
        f.gsc_clicks,
        f.gsc_impressions
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet' f
    LEFT JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' d
        ON f.content_hash_id = d.content_hash_id
    WHERE f.gsc_data_available IS TRUE
    LIMIT 100000
""").df()

print(features_df.shape)
print(features_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(100000, 9)
            content_hash_id report_date  gsc_avg_position  word_count  \
0  content_2e6360ad20fd7107  2026-03-01          3.500000        2855   
1  content_ac8663da7484669a  2026-03-01          5.875000        3281   
2  content_39d7361b4945d504  2026-03-01          4.333333        3579   
3  content_d49a012dcb924e31  2026-03-01          1.400000        2993   
4  content_614baf2af4330bd7  2026-03-01          3.476190        3500   

      content_type    main_intent  search_volume  gsc_clicks  gsc_impressions  
0  keyword article  informational             10           0                4  
1  keyword article  informational             10           0                8  
2  keyword article  informational             10           0               12  
3  keyword article  informational             10           0                5  
4  keyword article  informational             10           0               21  


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pandas as pd

# -----------------------------
# Copy dataframe
# -----------------------------
df = features_df.copy()

# -----------------------------
# Required columns
# -----------------------------
required_cols = [
    'gsc_clicks',
    'gsc_impressions',
    'gsc_avg_position',
    'word_count'
]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")

# -----------------------------
# Convert numeric columns
# -----------------------------
for col in required_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# -----------------------------
# Clean data
# -----------------------------
df = df.dropna(subset=required_cols)

df = df[df['gsc_impressions'] > 0]

# -----------------------------
# CTR
# -----------------------------
df['ctr'] = df['gsc_clicks'] / df['gsc_impressions']

print("Rows:", len(df))
print("Unique CTR values:", df['ctr'].nunique())

# -----------------------------
# Create binary label
# -----------------------------
threshold = df['ctr'].median()

df['low_ctr'] = (df['ctr'] <= threshold).astype(int)

print("\nClass Distribution")
print(df['low_ctr'].value_counts())

# Safety check
if df['low_ctr'].nunique() < 2:
    raise ValueError(
        "Only one class was created. Check CTR values."
    )

# -----------------------------
# Honest model
# -----------------------------
honest_features = [
    'gsc_avg_position',
    'word_count'
]

X_honest = df[honest_features]
y = df['low_ctr']

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model_honest = LogisticRegression(max_iter=1000)

model_honest.fit(X_train, y_train)

honest_score = model_honest.score(X_test, y_test)

print("\n" + "="*50)
print("HONEST MODEL")
print("="*50)
print(f"Accuracy: {honest_score:.4f}")

# -----------------------------
# Leakage demonstration
# -----------------------------
df['leaky_clicks'] = df['gsc_clicks']

leaky_features = honest_features + ['leaky_clicks']

X_leaky = df[leaky_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model_leaky = LogisticRegression(max_iter=1000)

model_leaky.fit(X_train, y_train)

leaky_score = model_leaky.score(X_test, y_test)

print("\n" + "="*50)
print("LEAKY MODEL")
print("="*50)
print(f"Accuracy: {leaky_score:.4f}")

print("\n" + "="*50)
print("LEAKAGE EFFECT")
print("="*50)
print(f"Honest Score : {honest_score:.4f}")
print(f"Leaky Score  : {leaky_score:.4f}")
print(f"Jump         : {leaky_score - honest_score:.4f}")

Rows: 70886
Unique CTR values: 2235

Class Distribution
low_ctr
1    60099
0    10787
Name: count, dtype: int64

HONEST MODEL
Accuracy: 0.8478

LEAKY MODEL
Accuracy: 1.0000

LEAKAGE EFFECT
Honest Score : 0.8478
Leaky Score  : 1.0000
Jump         : 0.1522


### Leakage Experiment Results

A binary target called `low_ctr` was created from CTR values.

Two logistic regression models were trained:

1. Honest model:
   - Features: `gsc_avg_position`, `word_count`
   - Accuracy: 0.8478

2. Leaky model:
   - Features: `gsc_avg_position`, `word_count`, `gsc_clicks`
   - Accuracy: 1.0000

The leaky model achieved perfect accuracy because `gsc_clicks` is directly used to compute CTR, which is then used to create the target variable.

This demonstrates target leakage. The model gains access to information that would not be available at prediction time and therefore produces unrealistically high performance.

Conclusion:
Models should only use features that are available before the target is observed. Any feature derived from the target or strongly dependent on it can invalidate evaluation results.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data Limits and Interpretation

### Dataset Limitations

1. **Single Time Period**
   - The analysis uses data from a single monthly partition (March 2026).
   - Results may not generalize to other months due to seasonality, content updates, or search algorithm changes.

2. **Observational Data**
   - The dataset records what happened but does not explain why it happened.
   - We can identify correlations but cannot prove causation.

3. **Position Bias**
   - Search position strongly influences CTR.
   - A page may receive low CTR because it ranks poorly rather than because the content is weak.

4. **Missing Business Context**
   - Metrics such as content quality, brand reputation, SERP features, and competitor activity are not captured.
   - These factors can significantly affect CTR and traffic.

5. **Aggregation Effects**
   - Data is aggregated at the page level.
   - Individual user behavior and search intent variations are not visible.

6. **Potential Measurement Noise**
   - Pages with very low impression counts may have unstable CTR values.
   - Small changes in clicks can create large percentage swings.

### Interpretation Guidelines

The analyses performed in this notebook are useful for:

- Identifying pages that may deserve optimization.
- Finding patterns associated with higher or lower CTR.
- Demonstrating feature engineering and leakage detection concepts.

The analyses should **not** be used to:

- Claim that a feature directly causes CTR changes.
- Estimate business impact with certainty.
- Predict future performance without additional validation on new data.

### Key Takeaway

This dataset supports exploratory analysis and opportunity discovery, but any recommendations should be validated through experimentation, additional time periods, and business review before implementation.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.